# 🧠 Sentiment Analysis (Duygu Analizi)

Bu notebook'ta **TextBlob** kütüphanesini kullanarak çektiğimiz alıntıların "duygusunu" ölçeceğiz.

**Sentiment Polarity (Duygu Kutupluluğu):**
* **-1.0**: Çok Negatif (Üzgün, Kızgın)
* **0.0**: Nötr
* **+1.0**: Çok Pozitif (Mutlu, Umutlu)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from textblob import TextBlob
import json
import os

# Veriyi tekrar yükleyelim (Kod tekrarı olmaması için)
json_path = '../../02-bs4-requests/scraped_data/quotes_structured.json'

if os.path.exists(json_path):
    with open(json_path, 'r', encoding='utf-8') as f:
        raw_data = json.load(f)
    if 'quotes' in raw_data:
        df = pd.DataFrame(raw_data['quotes'])
    else:
        df = pd.DataFrame(raw_data)
else:
    # Fallback dummy data
    df = pd.DataFrame([
        {"text": "I love this beautiful world!", "author": "Optimist"},
        {"text": "This is terrible and sad.", "author": "Pessimist"},
        {"text": "The book is on the table.", "author": "Neutral"}
    ])

print(f"Analiz edilecek veri sayısı: {len(df)}")

## 1. Duygu Skorlaması

In [ ]:
# Her bir alıntı için duygu analizi yap
def get_sentiment(text):
    blob = TextBlob(str(text))
    return blob.sentiment.polarity

# 'sentiment' sütunu oluştur
df['sentiment'] = df['text'].apply(get_sentiment)

# Örnek sonuçlar
df[['text', 'sentiment']].head()

## 2. Sonuçları Görselleştirme

In [ ]:
# Histogram: Duygu Dağılımı
plt.figure(figsize=(10, 6))
sns.histplot(df['sentiment'], bins=20, kde=True, color='purple')
plt.title('Alıntıların Duygu Dağılımı (-1: Negatif, +1: Pozitif)')
plt.xlabel('Sentiment Score')
plt.axvline(0, color='black', linestyle='--') # Nötr çizgisi
plt.show()

In [ ]:
# En Pozitif ve En Negatif Alıntılar
print("🌟 EN POZİTİF ALINTI:")
print(df.loc[df['sentiment'].idxmax()]['text'])
print(f"Skor: {df['sentiment'].max()}")

print("\n💔 EN NEGATİF ALINTI:")
print(df.loc[df['sentiment'].idxmin()]['text'])
print(f"Skor: {df['sentiment'].min()}")

## 3. Yazarlara Göre Mutluluk Analizi
Hangi yazar daha pozitif şeyler söylemiş?

In [ ]:
# Yazarlara göre ortalama sentiment skoru
author_sentiment = df.groupby('author')['sentiment'].mean().sort_values(ascending=False).head(10)

plt.figure(figsize=(12, 6))
sns.barplot(x=author_sentiment.values, y=author_sentiment.index, palette='coolwarm')
plt.title('Yazarların Ortalama Pozitiflik Skoru')
plt.xlabel('Ortalama Sentiment')
plt.axvline(0, color='black', linestyle='--')
plt.show()